In [ ]:
# === Colab bootstrap (no-op outside Colab) ===========================
# Clones the repo, installs minimal deps, mounts Google Drive, and
# symlinks heavy assets from Drive into the paths the notebook uses.
# See COLAB_SETUP.md for details.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, subprocess, sys
    REPO = "/content/INF8225_Projet"
    if not os.path.isdir(REPO):
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/Azcatchi17/INF8225_Projet.git", REPO,
        ])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()
# ======================================================================


In [1]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from MedSAM.segment_anything import sam_model_registry
from MedSAM.MedSAM_Inference import medsam_inference
from skimage import io, transform
from tqdm import tqdm

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
medsam_model = sam_model_registry["vit_b"](checkpoint="MedSAM/work_dir/MedSAM/medsam_vit_b.pth")
medsam_model = medsam_model.to(device)
medsam_model.eval()

Sam(
  (image_encoder): ImageEncoderViT(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): Linear(in_features=3072, out_features=768, bias=True)
          (act): GELU(approximate='none')
        )
      )
    )
    (neck): Sequential(
      (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): LayerNorm2d()
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (3): LayerNorm2d()
    )


In [3]:
def calculate_dice(mask_true, mask_pred):
    m_true = np.asarray(mask_true).astype(bool)
    m_pred = np.asarray(mask_pred).astype(bool)

    if m_true.sum() + m_pred.sum() == 0:
        return 1.0
    
    intersection = np.logical_and(m_true, m_pred).sum()
    return 2 * intersection / (m_true.sum() + m_pred.sum())

In [ ]:
# Boucle sur les images du dataset

img_folder = "data/Kvasir-SEG/images"
output_folder = "data/outputs"
masks_folder = "data/Kvasir-SEG/masks"

dice_list = []

with open("data/Kvasir-SEG/kavsir_bboxes.json") as f:
        bboxes = json.load(f)

for img in tqdm(os.listdir(img_folder), desc="Segmentation MedSAM"):
    img_path = os.path.join(img_folder, img)
    img_id = os.path.splitext(img)[0]

    img_np = io.imread(img_path)
    if len(img_np.shape) == 2:
        img_3c = np.repeat(img_np[:, :, None], 3, axis=-1)
    else:
        img_3c = img_np
    H, W, _ = img_3c.shape

    img_1024 = transform.resize(
        img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
    ).astype(np.uint8)
    img_1024 = (img_1024 - img_1024.min()) / np.clip(
        img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
    )  # normalize to [0, 1], (H, W, 3)
    # convert the shape to (3, H, W)
    img_1024_tensor = (
        torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)
    )

    with torch.no_grad():
            image_embedding = medsam_model.image_encoder(img_1024_tensor)  # (1, 256, 64, 64)

    full_medsam_seg = np.zeros((H, W), dtype=np.uint8)

    for bbox in bboxes[img_id]["bbox"]:
        x_min, y_min = bbox["xmin"], bbox["ymin"]
        x_max, y_max = bbox["xmax"], bbox["ymax"]
    
        box_np = np.array([[x_min, y_min, x_max, y_max]]) 
        # transfer box_np t0 1024x1024 scale
        box_1024 = box_np / np.array([W, H, W, H]) * 1024

        medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)

        full_medsam_seg[medsam_seg > 0] = 1

    io.imsave(
        os.path.join(output_folder, "seg_" + os.path.basename(img_path)),
        (full_medsam_seg*255).astype(np.uint8),
        check_contrast=False,
    )

    mask_path = os.path.join(masks_folder, img)
    true_seg = io.imread(mask_path)[:,:,0]
    true_seg[true_seg > 0] = 1

    dice_score = calculate_dice(true_seg, full_medsam_seg)

    dice_list.append({
    "image_id": img_id,
    "dice": dice_score
})
    
df = pd.DataFrame(dice_list)
df.to_csv("data/results/dice_medsam.csv", index=False)

Segmentation MedSAM: 100%|██████████| 1000/1000 [28:34<00:00,  1.71s/it]
